[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GNash19/paminaw/blob/v1.0.0/notebooks/solutions/02_transcribe.ipynb)

# Paminaw! · Lab 2 · Transcribe (solutions)

A standalone copy of one lab from the PyCon Davao 2026 workshop *Speech Processing Using
PyTorch* (Nash Golosino and Aurora Manseras), for study after the event. Run Whisper (and MMS on a GPU) on Tagalog and Cebuano.

Run **cell 0** first, then go from top to bottom with **Shift+Enter**. Scores are on the
**workshop subset (100 clips per language)**, not the paper's full-split numbers.

## 0 · Setup: run me first

This cell installs the workshop package `paminaw`, prints the versions of the libraries we
use, checks for a GPU and downloads the 200 audio clips. Wait for the **✅ READY** line
(about a minute). If something fails, the message tells you how to fix it.

In [ ]:
# Cell 0 · setup
import os
import subprocess
import sys

PAMINAW_REF = "v1.0.0"  # the workshop release on GitHub
SMOKE_TEST = os.environ.get("PAMINAW_SMOKE") == "1"  # facilitators: 2 clips, whisper-tiny

if "google.colab" in sys.modules:
    print(f"Installing paminaw {PAMINAW_REF} ...")
    url = f"paminaw @ git+https://github.com/GNash19/paminaw@{PAMINAW_REF}"
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", url],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-1500:])
        raise SystemExit("❌ Could not install paminaw. Fix: check that Colab says "
                         "'Connected' (top right), then run this cell again.")
try:
    import paminaw
except ModuleNotFoundError:
    raise SystemExit("❌ paminaw is not installed. Fix: outside Colab, run "
                     "`pip install -e .` in the repo folder (see docs/OFFLINE.md).") from None

SETUP = paminaw.setup_workshop(smoke=SMOKE_TEST)

## 2 · Transcribe (about 30 minutes)

A speech recogniser (ASR) works in stages:
**audio** → 16 kHz tensor → **log-mel features** → **encoder** (turns the picture into a
sequence of vectors) → **decoder** (writes text one token at a time) → **text**.

**Whisper** (OpenAI) is an encoder-decoder trained on 680,000 hours of audio from the web.
Its vocabulary has a language token for Tagalog, `<|tl|>`, but **none for Cebuano**.

**Why PyTorch rather than TensorFlow?** Both can run speech models. But the open speech
models we use (Whisper, MMS and SeamlessM4T) are published as PyTorch weights. Hugging
Face `transformers` v5 supports PyTorch only. And `torchaudio` gives us audio loading and
features in the same tensor library.

In [ ]:
import os

try:
    import paminaw
except ModuleNotFoundError:
    raise SystemExit("❌ Run cell 0 (Setup) at the top of the notebook first.") from None

# Lab 2 setup: safe to run even if you skipped the cells above
import time
from pathlib import Path

import pandas as pd
import torch
from transformers import pipeline

from paminaw.asr import pick_device_dtype

SMOKE_TEST = os.environ.get("PAMINAW_SMOKE") == "1"
paminaw.set_seed()
device, dtype = pick_device_dtype()  # ("cuda", float16) on a GPU, ("cpu", float32) otherwise
WHISPER_MODEL = "openai/whisper-tiny" if SMOKE_TEST else "openai/whisper-small"

# 👉 The one number to change: clips per language. 20 on CPU (~5 min), all 100 on a GPU.
N_CLIPS = 2 if SMOKE_TEST else (100 if device == "cuda" else 20)

tl = paminaw.load_subset("tagalog", n=N_CLIPS)
ceb = paminaw.load_subset("cebuano", n=N_CLIPS)
print(f"{WHISPER_MODEL} on {device} ({str(dtype).replace('torch.', '')}), "
      f"{N_CLIPS} clips per language")

In [ ]:
# Download Whisper (about 1 GB for whisper-small; Colab's network is fast) and build a pipeline.
asr = pipeline("automatic-speech-recognition", model=WHISPER_MODEL, device=device, dtype=dtype)

In [ ]:
wav = paminaw.load_16k_mono(tl.path[0])

with torch.inference_mode():  # we only predict, so skip gradient bookkeeping
    out = asr({"raw": wav.squeeze(0).numpy(), "sampling_rate": 16_000},
              generate_kwargs={"language": "tagalog", "task": "transcribe"})

print("REFERENCE :", tl.transcription[0])
print("WHISPER   :", out["text"])

### Transcribe all clips

We pass `language="tagalog"` for the Cebuano clips too. Whisper has no Cebuano option, so
Tagalog is the nearest language it is allowed to write; the paper calls this the
*language-committed* condition. Results are saved to `outputs/` for Lab 3.

In [ ]:
rows, t0, audio_seconds = [], time.perf_counter(), 0.0
for lang, clips in [("tagalog", tl), ("cebuano", ceb)]:
    for c in clips.itertuples():
        wav = paminaw.load_16k_mono(c.path)
        with torch.inference_mode():
            text = asr({"raw": wav.squeeze(0).numpy(), "sampling_rate": 16_000},
                       generate_kwargs={"language": "tagalog", "task": "transcribe"})["text"]
        rows.append({"lang": lang, "idx": c.idx, "fleurs_id": c.fleurs_id,
                     "hypothesis": text, "reference": c.transcription})
        audio_seconds += c.duration_s
        print(f"\r{lang}: {len(rows)} clips done", end="")
seconds = time.perf_counter() - t0
print(f"\nTranscribed {audio_seconds / 60:.1f} min of audio in {seconds:.0f} s "
      f"(real-time factor {seconds / audio_seconds:.2f}).")

mine = pd.DataFrame(rows)
Path("outputs").mkdir(exist_ok=True)
mine.to_csv("outputs/whisper_small_yours.csv", index=False)
pd.set_option("display.max_colwidth", 90)
mine[["lang", "reference", "hypothesis"]].groupby("lang").head(3)

### ✏️ Exercise 4 · Which language does Whisper hear?

Remove the language hint on a Cebuano clip and let Whisper decide. 🤔 Before you run it:
what do you expect it to detect? Then read the **language token** Whisper chose.

In [ ]:
wav = paminaw.load_16k_mono(ceb.path[0])
audio_in = {"raw": wav.squeeze(0).numpy(), "sampling_rate": 16_000}

with torch.inference_mode():
    no_hint = asr(audio_in, generate_kwargs={"task": "transcribe"})  # no "language" key
print("REFERENCE (Cebuano):", ceb.transcription[0])
print("NO HINT            :", no_hint["text"])

# Ask Whisper directly which language token it would start with
features = asr.feature_extractor(audio_in["raw"], sampling_rate=16_000, return_tensors="pt")
with torch.inference_mode():
    lang_ids = asr.model.detect_language(features.input_features.to(device, dtype))
print("Detected language token:", asr.tokenizer.convert_ids_to_tokens(lang_ids.tolist()))
print("Does Whisper even have a Cebuano token?", "<|ceb|>" in asr.tokenizer.get_vocab())

In [ ]:
# Solution: check all Cebuano clips, and compare with Whisper large-v3 from the paper's runs
detected = []
for p in ceb.path:
    w = paminaw.load_16k_mono(p).squeeze(0).numpy()
    f = asr.feature_extractor(w, sampling_rate=16_000, return_tensors="pt").input_features
    with torch.inference_mode():
        ids = asr.model.detect_language(f.to(device, dtype))
    detected += asr.tokenizer.convert_ids_to_tokens(ids.tolist())
print(f"{WHISPER_MODEL}, {len(detected)} Cebuano clips:", pd.Series(detected).value_counts().to_dict())

large = paminaw.load_precomputed("Whisper large-v3")
print("Whisper large-v3 (paper run), 100 Cebuano clips:",
      large[large.lang == "cebuano"]["detected_language"].value_counts().to_dict())

**Answer.** Whisper detects **Tagalog** (`<|tl|>`). It has no Cebuano token at all, so the
nearest language it is allowed to write is the related Central Philippine language it
knows. In the paper, Whisper large-v3 chose Tagalog for **every one** of the 541 Cebuano
test clips, and the "with hint" and "no hint" runs came out byte-identical. Its Cebuano
errors are largely Tagalog words: Cebuano *nga* written as Tagalog *na*/*ng*, *tawo* as
*tao*, *dili* as *hindi*. Smaller Whisper models sometimes pick another language on a
clip, but never Cebuano.

## 2b · MMS-1B-all: a model that *does* know Cebuano (GPU runtimes)

**MMS** (Massively Multilingual Speech, Meta) is one shared wav2vec 2.0 model plus a small
**adapter** for each of 1,162 languages, including Cebuano (`ceb`) and Tagalog (`tgl`).
It is a **CTC** model: instead of writing word by word, it scores every letter (or a
"blank") for each 20 ms frame. We take the most likely one per frame (**argmax**), then
merge repeats and drop blanks.

### Three choices every time you run a model

| Choice | What we use | Why |
|---|---|---|
| **Device** | `cuda` if there is a GPU, else `cpu` | A GPU does thousands of multiplications at once. MMS has a billion parameters, so it only runs live on a GPU here. |
| **Precision** | `float16` for Whisper on a GPU; `float32` for MMS and anything on a CPU | Half precision halves memory and speeds up GPUs, but CPUs do not benefit. The MMS adapters are float32 weights, so the paper runs MMS in float32. |
| **`torch.inference_mode()`** | always, when transcribing | Training has to remember every step to compute gradients; inference does not. Switching that off saves memory and time. |

On a CPU runtime this lab loads the paper's precomputed MMS outputs for the same clips
instead.

In [ ]:
import os

try:
    import paminaw
except ModuleNotFoundError:
    raise SystemExit("❌ Run cell 0 (Setup) at the top of the notebook first.") from None

import pandas as pd
import torch
from transformers import AutoProcessor, Wav2Vec2ForCTC

SMOKE_TEST = os.environ.get("PAMINAW_SMOKE") == "1"
paminaw.set_seed()
RUN_MMS = torch.cuda.is_available() and not SMOKE_TEST
N_MMS = 100 if RUN_MMS else 20
clips = {lang: paminaw.load_subset(lang, n=N_MMS) for lang in ("cebuano", "tagalog")}

if RUN_MMS:
    processor = AutoProcessor.from_pretrained("facebook/mms-1b-all")
    mms = Wav2Vec2ForCTC.from_pretrained("facebook/mms-1b-all", dtype=torch.float32).to("cuda")
    rows = []
    for lang, code in [("cebuano", "ceb"), ("tagalog", "tgl")]:
        processor.tokenizer.set_target_lang(code)  # the adapter's alphabet
        mms.load_adapter(code)  # swap in the language adapter (~2M parameters)
        mms.to("cuda")
        for c in clips[lang].itertuples():
            wav = paminaw.load_16k_mono(c.path).squeeze(0).numpy()
            inputs = processor(wav, sampling_rate=16_000, return_tensors="pt").to("cuda")
            with torch.inference_mode():
                logits = mms(**inputs).logits
            ids = torch.argmax(logits, dim=-1)[0]  # best letter per 20 ms frame
            rows.append({"lang": lang, "idx": c.idx, "hypothesis": processor.decode(ids)})
    mms_out = pd.DataFrame(rows)
    print(f"✅ MMS ran live on {torch.cuda.get_device_name(0)} for {len(mms_out)} clips.")
else:
    print("ℹ️ No GPU (or smoke test): loading the paper's precomputed MMS-1B-all outputs "
          "for the same clips instead.")
    pre = paminaw.load_precomputed("MMS-1B-all")
    keys = pd.concat(clips.values())[["lang", "idx"]]
    mms_out = pre.merge(keys, on=["lang", "idx"])[["lang", "idx", "hypothesis"]]

ref = pd.concat(clips.values())[["lang", "idx", "transcription"]]
show = ref.merge(mms_out, on=["lang", "idx"]).rename(columns={"transcription": "reference"})
pd.set_option("display.max_colwidth", 90)
show[show.lang == "cebuano"].head(3)[["reference", "hypothesis"]]